# ДЗ №3. Ранжирование на основе datamart

Общая информация
Дата выдачи: 3 апреля 2026

Дедлайн: 26 апреля 2026 23:59 MSK

В этом домашнем задании мы продолжим строить приближенную к реальной рекомендательную систему. Работать будем с данными marketplace из [T-ECD](https://huggingface.co/datasets/t-tech/T-ECD).

Обычно рекомендательная система состоит из нескольких этапов:
1. Отбор кандидатов (Retrieval)
2. Ранжирование (Ranking)
3. Бизнес-логика (например, условие на то, чтобы товары от одного продавца не стояли в ленте друг за другом)

В этом домашнем задании сосредоточимся на втором этапе. Можно и нужно использовать наработки из предыдущего домашнего задания!

Краткое напоминание, почему отбор кандидатов и ранжирование - разные этапы. Задачу рекомендаций можно решать как регрессию (насколько релевантен айтем), классификацию (релевантен ли айтем) или ранжирование (какой из двух айтемов релевантнее). В идеале - проранжировать каталог под каждого пользователя. Но каталог всегда существенно больше того подмножества айтемов, которые пользователь увидит в итоговой выдаче. А качественно ранжировать весь каталог - ОЧЕНЬ долго и дорого. Получаем trade-off скорости и качества. Простое решение - многостадийные рекомендации. Сначала отберем кандидатов (релевантные/не релевантные), а потом проранжируем только релевантные.

В этом задании следующая разбалловка:

1) Cбор датамарта - 4 балла
2) Сбор датасета для обучения ранжирования - 2 балла
3) Сбор град. бустинга и оценка по метрикам с бейзлайном - 4 балла

Соответственно, максимум можно набрать 10 баллов.

In [1]:
!pip install -q polars lightgbm scikit-learn catboost shap optuna implicit torch

In [1]:
import json
import gc
import os
import random
import typing as t
from abc import ABC, abstractmethod
from collections import defaultdict
from dataclasses import dataclass
from functools import partial
from pathlib import Path

import joblib
import lightgbm as lgbm
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import polars as pl
import polars.selectors as cs
import shap
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import HTML
from implicit.als import AlternatingLeastSquares
from optuna.samplers import TPESampler
from scipy.sparse import coo_matrix, csr_matrix
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Dataset, IterableDataset

/Users/maksimlunin/Desktop/jupyter/Рекомендательные системы/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Download Data

Данные занимают около 3.5 GB

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="t-tech/T-ECD",
    repo_type="dataset",
    local_dir=".",
    local_dir_use_symlinks=False,
    allow_patterns=["dataset/small/users.pq", "dataset/small/marketplace/**"]
)

/Users/maksimlunin/Desktop/jupyter/Рекомендательные системы/venv/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Fetching ... files: 229it [01:19,  2.88it/s]


'/Users/maksimlunin/Desktop/jupyter/Рекомендательные системы'

Больше всего места занимают эмбеддинги айтемов, их по желанию можно удалить, так как в этой работе нам потребуется дополнительное место на создание датамарта

In [3]:
pl.read_parquet("dataset/small/marketplace/items.pq").drop("embedding").write_parquet("dataset/small/marketplace/items.pq")

## I. Datamart (4 балла)

В задаче ранжирования хорошо себя показывают бустинги ([Catboost](https://catboost.ai), [LGBM](https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.Booster.html), [XGBoost](https://xgboost.readthedocs.io/en/stable/)). С точки зрения интерфейса: Algorithm(user, item, [features]), где фичи - любые полезные статистики (количество просмотров, конверсия из клика в кликаут, средний рейтинг, ...).

Каждый раз рассчитывать фичи с нуля по сырым логам достаточно тяжело (а в реальности рекомендации мы делаем не один раз в домашней работе, а гораздо чаще). Учитывая, что логи могут иметь разный формат или нуждаться в  дополнительной фильтрации (баги в логах всё же не редкость). Простое решение - предподсчитать статистики и сохранить их в отдельных файлах, которые затем удобно просто прочитать. 

Это удобно сделать посредством датамарта. Датамарт - это витрина данных под определенную задачу. Он состоит из слоёв, где переход между слоями задает преобразование над данными, и обычно такие преобразования выполняются раз в какое-то время (в нашем случае пусть будет день). Для задачи ранжирования нам потребуется три слоя: 
1. Raw - содержит сырые логи за каждый день. Мы уже собрали его на предыдущем шаге.
2. Aggs - содержит агрегированные статистики по юзерам и айтемам за каждый день (user, item, [stats]). Например, количество просмотров, количество кликов, количество кликов c поверхности поиска.
3. Features - содержит фичи, которые мы хотим использовать в модели, тоже за каждый день. Например, средний рейтинг айтема, средний рейтинг айтема по категориям, общая конверсия из клика в кликаут для пользователя за последние 30 дней. На этом слое удобно выделить отдельные папки по группам фичей (user, item, user-item, ...), чтобы избежать дубликатов при хранении. 

Возьмем только небольшой срез данных, иначе дальнейшая работа может стать computationally infeasible. Переложим этот срез в `datamart/raw/events/{action_type}/{day}.pq`

Именно в таком формате логи обычно хранятся в сервисе. 

Вы можете расширить условия на сэмплирование юзеров и айтемов. Если у вас будут проблемы с памятью, то можете и уменьшить что-то, но чем меньше ваш датасет, тем хуже будут метрики у конечной модели

In [4]:
ACTION_TYPES = ["view", "click", "clickout", "like"]
SUBDOMAINS = ["u2i", "i2i", "catalog", "search", "other"]

DAYS = list(range(1250, 1301))

Будем считать, что view < click < clickout < like с точки зрения бизнеса. Этот факт будет использоваться далее.

In [5]:
selected_users = (
    pl.concat(
        [pl.scan_parquet(f"dataset/small/marketplace/events/{str(day).zfill(5)}.pq") for day in DAYS[-10:]]
    )
    .group_by("user_id").agg(pl.len()).sort("len", descending=True)
    .head(20000).collect()["user_id"].to_list()
)

selected_items = (
    pl.concat(
        [pl.scan_parquet(f"dataset/small/marketplace/events/{str(day).zfill(5)}.pq") for day in DAYS[-10:]]
    ).group_by("item_id").agg(pl.len()).sort("len", descending=True)
    .head(20000).collect()["item_id"].to_list()
)

In [6]:
USERS = pl.scan_parquet("dataset/small/users.pq").filter(pl.col("user_id").is_in(selected_users)).collect()
print(USERS.shape)
ITEMS = pl.scan_parquet("dataset/small/marketplace/items.pq").filter(pl.col("item_id").is_in(selected_items)).collect()

(20000, 3)


### I.I. Datamart -> Raw слой (1 из 4 баллов)

Здесь вам надо собрать  raw слой:

![](images/datamart_raw.png)

в каждом файлике должны храниться данные на конкретную дату


In [7]:
raw_events_dir = Path("datamart/raw/events")

for action_type in ACTION_TYPES:
    os.makedirs(raw_events_dir / action_type, exist_ok=True)

for day in tqdm(DAYS):
    day_str = str(day).zfill(5)
    day_events = (
        pl.scan_parquet(f"dataset/small/marketplace/events/{day_str}.pq")
        .filter(
            pl.col("user_id").is_in(selected_users)
            & pl.col("item_id").is_in(selected_items)
            & pl.col("action_type").is_in(ACTION_TYPES)
            & pl.col("subdomain").is_in(SUBDOMAINS)
        )
    )

    for action_type in ACTION_TYPES:
        (
            day_events
            .filter(pl.col("action_type") == action_type)
            .collect()
            .write_parquet(raw_events_dir / action_type / f"{day_str}.pq")
        )

100%|██████████| 51/51 [00:04<00:00, 12.12it/s]


### I.II. Datamart -> Agg слой (1 из 4 баллов)

Рассчитате количество событий каждого типа (`action_type`) по каждой поверхности (`subdomain`) по парам (`user_id`, `item_id`) за каждый день. Не забудьте про общий счетчик - сумму по всем поверхностям. Сохраните в виде polars-таблиц.

Аналогично raw, но в agg значения внутри дня должны быть агрегированы

![](images/datamart_agg.png)

In [8]:
events_dir = Path("datamart/aggs/events/")
os.makedirs(events_dir, exist_ok=True)

for day in tqdm(DAYS):
    day_str = str(day).zfill(5)

    day_df = pl.concat([
        pl.scan_parquet(f"datamart/raw/events/{action_type}/{day_str}.pq")
        for action_type in ACTION_TYPES
    ])

    agg_exprs = []
    for action_type in ACTION_TYPES:
        agg_exprs.append(
            pl.col("action_type")
            .eq(action_type)
            .sum()
            .cast(pl.Float32)
            .alias(f"num_{action_type}_all_subdomains")
        )
        for subdomain in SUBDOMAINS:
            agg_exprs.append(
                (
                    pl.col("action_type").eq(action_type)
                    & pl.col("subdomain").eq(subdomain)
                )
                .sum()
                .cast(pl.Float32)
                .alias(f"num_{action_type}_{subdomain}")
            )

    (
        day_df
        .group_by(["user_id", "item_id"])
        .agg(agg_exprs)
        .collect()
        .write_parquet(events_dir / f"{day_str}.pq")
    )

100%|██████████| 51/51 [00:00<00:00, 71.40it/s] 


Пример того, что может получиться.

In [9]:
pl.read_parquet("datamart/aggs/events/01300.pq").sample(10)

user_id,item_id,num_view_all_subdomains,num_view_u2i,num_view_i2i,num_view_catalog,num_view_search,num_view_other,num_click_all_subdomains,num_click_u2i,num_click_i2i,num_click_catalog,num_click_search,num_click_other,num_clickout_all_subdomains,num_clickout_u2i,num_clickout_i2i,num_clickout_catalog,num_clickout_search,num_clickout_other,num_like_all_subdomains,num_like_u2i,num_like_i2i,num_like_catalog,num_like_search,num_like_other
u64,str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
86774717,"""nfmcg_1992136""",1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19972060,"""nfmcg_25771817""",1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6101160,"""nfmcg_19624867""",1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
55543320,"""nfmcg_2826310""",1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9510427,"""nfmcg_1675674""",1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
35267328,"""nfmcg_26956761""",1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19264857,"""nfmcg_6458588""",1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
23881693,"""nfmcg_23458520""",1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
43356997,"""nfmcg_10382568""",1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### I.III. Datamart -> Feature слой (2 из 4 баллов)

Подумайте, какие признаки, рассчитанные на основе ранее собранных статистик, можно будет использовать в качестве фичей для модели. Реализуйте логику их подсчета за каждый день. Обратите внимание, что все фичи необходимо рассчитывать по какому-то временному окну, например "количество кликов на товар за последние 14 дней".

Сохраните признаки в виде polars-таблиц. Не забудьте про декомпозицию на отдельные папки по сущностям с целью избежать дубликатов при хранении.

Минимально должно получиться 10 признаков, из которых:
* 2 должны относиться только к сущности "пользователь" (например, медианная цена кликнутых айтемов у этого пользователя)
* 2 должны относиться только к сущности "айтем" (например, средняя конверсия из клика в кликаут по поверхности "поиск" у этого айтема)
* 6 признаков, которые показывают связь пользователя и айтема (например, количество просмотров этого айтема у этого пользователя).

Однако настоятельно рекомендуется собрать больше признаков. В данных много сущностей - категория, соцдем-кластер, бренд. И много параметров - цена, поверхность, тип события. В том числе можно рассчитать "изменение признака относительно предыдущего дня". Используйте polars expressions для написания шаблонного кода, в который затем удобно подставить конкретные названия сущностей и параметров, и получить готовый набор признаков. 

P.S. Цену для товаров мы считаем фиксированной, она представлена в каталоге `ITEMS`

Может быть полезно посчитать конверсии как отношение числа второго события из пары к числу первого события из пары. Аккуратнее с делением на ноль - возможно, пользователей, для которых не определено число первого события из пары, не стоит учитывать при расчетах.

Пример того, как должно получиться:

![](images/datamart_feat2.png)

In [10]:
CONVERSION_PAIRS = [
    ("view", "click"), 
    ("view", "clickout"), 
    ("view", "like"),
    ("click", "clickout"),
    ("click", "like"),
    ("clickout", "like"),
]

Удобно называть фичи следующим образом.

In [11]:
def _feature_name(
    feature: str,
    keys: list[str],
    type: t.Literal["num", "cat"] = "num",
) -> str:
    return f"f_{type}__{'_'.join(keys)}__{feature}"

In [12]:
def calculate_user_group_item_group_features(
    day_from: int | None = None,
    day_to: int | None = None,
    num_days: int = 14,
    user_group: t.Literal["region", "socdem_cluster"] | None = None,
    item_group: t.Literal["brand_id", "category", "subcategory", "item_id"] = "item_id"
) -> None:
    global USERS, ITEMS

    if day_from is None:
        day_from = min(DAYS)
    if day_to is None:
        day_to = max(DAYS)

    join_user_cols = ["user_id"] + ([user_group] if user_group is not None else [])

    key_cols = ([user_group] if user_group is not None else []) + [item_group]
    out_dir = Path(f"datamart/features/events/{'-'.join(key_cols)}/")
    out_dir.mkdir(parents=True, exist_ok=True)

    users_lf = USERS.lazy().select(join_user_cols) if user_group is not None else None
    items_lf = None if item_group == "item_id" else ITEMS.lazy().select(["item_id", item_group])

    for day in tqdm(range(day_from, day_to + 1)):
        day_str = str(day).zfill(5)
        start_day = max(day_from, day - num_days + 1)
        window_days = [
            d for d in range(start_day, day + 1)
            if (Path("datamart/aggs/events") / f"{str(d).zfill(5)}.pq").exists()
        ]
        if not window_days:
            continue

        aggs_lf = pl.concat([
            pl.scan_parquet(f"datamart/aggs/events/{str(d).zfill(5)}.pq")
            for d in window_days
        ])

        base_df = aggs_lf
        if users_lf is not None:
            base_df = base_df.join(users_lf, on="user_id", how="left")
        if items_lf is not None:
            base_df = base_df.join(items_lf, on="item_id", how="left")

        base = (
            base_df
            .group_by(key_cols)
            .agg([
                *[
                    pl.col(f"num_{action}_all_subdomains").sum().alias(
                        _feature_name(f"num_{action}_{num_days}d", key_cols)
                    )
                    for action in ACTION_TYPES
                ],
                *[
                    pl.col(f"num_{action}_{subdomain}").sum().alias(
                        _feature_name(f"num_{action}_{subdomain}_{num_days}d", key_cols)
                    )
                    for action in ACTION_TYPES
                    for subdomain in SUBDOMAINS
                ],
            ])
        )

        conv_exprs = []
        for left_action, right_action in CONVERSION_PAIRS:
            num_col = _feature_name(f"num_{right_action}_{num_days}d", key_cols)
            den_col = _feature_name(f"num_{left_action}_{num_days}d", key_cols)
            conv_col = _feature_name(f"conv_{left_action}_to_{right_action}_{num_days}d", key_cols)
            conv_exprs.append(
                pl.when(pl.col(den_col) > 0)
                .then(pl.col(num_col) / pl.col(den_col))
                .otherwise(None)
                .cast(pl.Float32)
                .alias(conv_col)
            )

        (
            base
            .with_columns(conv_exprs)
            .collect()
            .write_parquet(out_dir / f"{day_str}.pq")
        )


def calculate_user_item_group_features(
    day_from: int | None = None,
    day_to: int | None = None,
    num_days: int = 14,
    item_group: t.Literal["item_id", "brand_id", "category", "subcategory"] | None = None
) -> None:
    global ITEMS

    if day_from is None:
        day_from = min(DAYS)
    if day_to is None:
        day_to = max(DAYS)

    if item_group is None:
        key_cols = ["user_id"]
        out_dir = Path("datamart/features/events/user_id/")
    else:
        key_cols = ["user_id", item_group]
        out_dir = Path(f"datamart/features/events/{'-'.join(key_cols)}/")

    out_dir.mkdir(parents=True, exist_ok=True)
    items_lf = (
        ITEMS.lazy().select(["item_id", item_group])
        if (item_group is not None and item_group != "item_id")
        else None
    )

    for day in tqdm(range(day_from, day_to + 1)):
        day_str = str(day).zfill(5)
        start_day = max(day_from, day - num_days + 1)
        window_days = [
            d for d in range(start_day, day + 1)
            if (Path("datamart/aggs/events") / f"{str(d).zfill(5)}.pq").exists()
        ]
        if not window_days:
            continue

        aggs_lf = pl.concat([
            pl.scan_parquet(f"datamart/aggs/events/{str(d).zfill(5)}.pq")
            for d in window_days
        ])

        if items_lf is not None:
            aggs_lf = aggs_lf.join(items_lf, on="item_id", how="left")

        base = aggs_lf.group_by(key_cols).agg([
            *[
                pl.col(f"num_{action}_all_subdomains").sum().alias(
                    _feature_name(f"num_{action}_{num_days}d", key_cols)
                )
                for action in ACTION_TYPES
            ],
            *[
                pl.col(f"num_{action}_{subdomain}").sum().alias(
                    _feature_name(f"num_{action}_{subdomain}_{num_days}d", key_cols)
                )
                for action in ACTION_TYPES
                for subdomain in SUBDOMAINS
            ],
        ])

        conv_exprs = []
        for left_action, right_action in CONVERSION_PAIRS:
            num_col = _feature_name(f"num_{right_action}_{num_days}d", key_cols)
            den_col = _feature_name(f"num_{left_action}_{num_days}d", key_cols)
            conv_col = _feature_name(f"conv_{left_action}_to_{right_action}_{num_days}d", key_cols)
            conv_exprs.append(
                pl.when(pl.col(den_col) > 0)
                .then(pl.col(num_col) / pl.col(den_col))
                .otherwise(None)
                .cast(pl.Float32)
                .alias(conv_col)
            )

        (
            base
            .with_columns(conv_exprs)
            .collect()
            .write_parquet(out_dir / f"{day_str}.pq")
        )

In [13]:
for item_group in [None, "item_id", "brand_id", "category"]:
    print(f"Calculating features for user_id and {item_group}")
    calculate_user_item_group_features(
        num_days=30, item_group=item_group
    )

Calculating features for user_id and None


100%|██████████| 51/51 [00:02<00:00, 20.66it/s]


Calculating features for user_id and item_id


100%|██████████| 51/51 [00:06<00:00,  7.96it/s]


Calculating features for user_id and brand_id


100%|██████████| 51/51 [00:03<00:00, 15.64it/s]


Calculating features for user_id and category


100%|██████████| 51/51 [00:04<00:00, 11.55it/s]


In [14]:
for user_group in [None, "socdem_cluster"]:
    for item_group in ["item_id", "brand_id", "category"]:
        print(f"Calculating features for {user_group} and {item_group}")
        calculate_user_group_item_group_features(
            num_days=30, user_group=user_group, item_group=item_group
        )

Calculating features for None and item_id


100%|██████████| 51/51 [00:03<00:00, 16.90it/s]


Calculating features for None and brand_id


100%|██████████| 51/51 [00:02<00:00, 17.55it/s]


Calculating features for None and category


...

100%|██████████| 51/51 [00:04<00:00, 12.14it/s]


Calculating features for socdem_cluster and brand_id


100%|██████████| 51/51 [00:03<00:00, 14.74it/s]


Calculating features for socdem_cluster and category


100%|██████████| 51/51 [00:04<00:00, 12.19it/s]


Пример того, что может получиться.

In [15]:
!ls datamart/features/events

brand_id                socdem_cluster-category user_id-category
category                socdem_cluster-item_id  user_id-item_id
item_id                 user_id
socdem_cluster-brand_id user_id-brand_id


In [16]:
pl.read_parquet("datamart/features/events/user_id/01250.pq").sample(10)

user_id,f_num__user_id__num_view_30d,f_num__user_id__num_click_30d,f_num__user_id__num_clickout_30d,f_num__user_id__num_like_30d,f_num__user_id__num_view_u2i_30d,f_num__user_id__num_view_i2i_30d,f_num__user_id__num_view_catalog_30d,f_num__user_id__num_view_search_30d,f_num__user_id__num_view_other_30d,f_num__user_id__num_click_u2i_30d,f_num__user_id__num_click_i2i_30d,f_num__user_id__num_click_catalog_30d,f_num__user_id__num_click_search_30d,f_num__user_id__num_click_other_30d,f_num__user_id__num_clickout_u2i_30d,f_num__user_id__num_clickout_i2i_30d,f_num__user_id__num_clickout_catalog_30d,f_num__user_id__num_clickout_search_30d,f_num__user_id__num_clickout_other_30d,f_num__user_id__num_like_u2i_30d,f_num__user_id__num_like_i2i_30d,f_num__user_id__num_like_catalog_30d,f_num__user_id__num_like_search_30d,f_num__user_id__num_like_other_30d,f_num__user_id__conv_view_to_click_30d,f_num__user_id__conv_view_to_clickout_30d,f_num__user_id__conv_view_to_like_30d,f_num__user_id__conv_click_to_clickout_30d,f_num__user_id__conv_click_to_like_30d,f_num__user_id__conv_clickout_to_like_30d
u64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
8062491,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
58539847,23.0,1.0,1.0,0.0,20.0,2.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.043478,0.043478,0.0,1.0,0.0,0.0
26912500,11.0,0.0,0.0,0.0,11.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
42638275,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
49806427,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
41475194,10.0,2.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0,0.0,0.0,null
54792045,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
22236826,5.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
9061186,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null


In [17]:
pl.read_parquet("datamart/features/events/item_id/01250.pq").sample(10)

item_id,f_num__item_id__num_view_30d,f_num__item_id__num_click_30d,f_num__item_id__num_clickout_30d,f_num__item_id__num_like_30d,f_num__item_id__num_view_u2i_30d,f_num__item_id__num_view_i2i_30d,f_num__item_id__num_view_catalog_30d,f_num__item_id__num_view_search_30d,f_num__item_id__num_view_other_30d,f_num__item_id__num_click_u2i_30d,f_num__item_id__num_click_i2i_30d,f_num__item_id__num_click_catalog_30d,f_num__item_id__num_click_search_30d,f_num__item_id__num_click_other_30d,f_num__item_id__num_clickout_u2i_30d,f_num__item_id__num_clickout_i2i_30d,f_num__item_id__num_clickout_catalog_30d,f_num__item_id__num_clickout_search_30d,f_num__item_id__num_clickout_other_30d,f_num__item_id__num_like_u2i_30d,f_num__item_id__num_like_i2i_30d,f_num__item_id__num_like_catalog_30d,f_num__item_id__num_like_search_30d,f_num__item_id__num_like_other_30d,f_num__item_id__conv_view_to_click_30d,f_num__item_id__conv_view_to_clickout_30d,f_num__item_id__conv_view_to_like_30d,f_num__item_id__conv_click_to_clickout_30d,f_num__item_id__conv_click_to_like_30d,f_num__item_id__conv_clickout_to_like_30d
str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""nfmcg_9355206""",3.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
"""nfmcg_11653972""",3.0,1.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.333333,0.0,0.0,0.0,0.0,null
"""nfmcg_9158926""",1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
"""nfmcg_3223037""",1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
"""nfmcg_7285162""",1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
"""nfmcg_2458649""",8.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
"""nfmcg_6521281""",1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
"""nfmcg_15951326""",1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
"""nfmcg_7810100""",4.0,0.0,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null


In [18]:
pl.read_parquet("datamart/features/events/user_id-item_id/01250.pq").sample(10)

user_id,item_id,f_num__user_id_item_id__num_view_30d,f_num__user_id_item_id__num_click_30d,f_num__user_id_item_id__num_clickout_30d,f_num__user_id_item_id__num_like_30d,f_num__user_id_item_id__num_view_u2i_30d,f_num__user_id_item_id__num_view_i2i_30d,f_num__user_id_item_id__num_view_catalog_30d,f_num__user_id_item_id__num_view_search_30d,f_num__user_id_item_id__num_view_other_30d,f_num__user_id_item_id__num_click_u2i_30d,f_num__user_id_item_id__num_click_i2i_30d,f_num__user_id_item_id__num_click_catalog_30d,f_num__user_id_item_id__num_click_search_30d,f_num__user_id_item_id__num_click_other_30d,f_num__user_id_item_id__num_clickout_u2i_30d,f_num__user_id_item_id__num_clickout_i2i_30d,f_num__user_id_item_id__num_clickout_catalog_30d,f_num__user_id_item_id__num_clickout_search_30d,f_num__user_id_item_id__num_clickout_other_30d,f_num__user_id_item_id__num_like_u2i_30d,f_num__user_id_item_id__num_like_i2i_30d,f_num__user_id_item_id__num_like_catalog_30d,f_num__user_id_item_id__num_like_search_30d,f_num__user_id_item_id__num_like_other_30d,f_num__user_id_item_id__conv_view_to_click_30d,f_num__user_id_item_id__conv_view_to_clickout_30d,f_num__user_id_item_id__conv_view_to_like_30d,f_num__user_id_item_id__conv_click_to_clickout_30d,f_num__user_id_item_id__conv_click_to_like_30d,f_num__user_id_item_id__conv_clickout_to_like_30d
u64,str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
65683354,"""nfmcg_20507923""",1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
78511529,"""nfmcg_37761""",1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
85998098,"""nfmcg_1576132""",4.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
43011128,"""nfmcg_22274857""",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
16624924,"""nfmcg_7763862""",1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
12381786,"""nfmcg_13513801""",1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
78824315,"""nfmcg_12977198""",1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
22417367,"""nfmcg_16396259""",1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
65180564,"""nfmcg_15241635""",1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null


In [19]:
pl.read_parquet("datamart/features/events/socdem_cluster-brand_id/01250.pq").sample(10)

socdem_cluster,brand_id,f_num__socdem_cluster_brand_id__num_view_30d,f_num__socdem_cluster_brand_id__num_click_30d,f_num__socdem_cluster_brand_id__num_clickout_30d,f_num__socdem_cluster_brand_id__num_like_30d,f_num__socdem_cluster_brand_id__num_view_u2i_30d,f_num__socdem_cluster_brand_id__num_view_i2i_30d,f_num__socdem_cluster_brand_id__num_view_catalog_30d,f_num__socdem_cluster_brand_id__num_view_search_30d,f_num__socdem_cluster_brand_id__num_view_other_30d,f_num__socdem_cluster_brand_id__num_click_u2i_30d,f_num__socdem_cluster_brand_id__num_click_i2i_30d,f_num__socdem_cluster_brand_id__num_click_catalog_30d,f_num__socdem_cluster_brand_id__num_click_search_30d,f_num__socdem_cluster_brand_id__num_click_other_30d,f_num__socdem_cluster_brand_id__num_clickout_u2i_30d,f_num__socdem_cluster_brand_id__num_clickout_i2i_30d,f_num__socdem_cluster_brand_id__num_clickout_catalog_30d,f_num__socdem_cluster_brand_id__num_clickout_search_30d,f_num__socdem_cluster_brand_id__num_clickout_other_30d,f_num__socdem_cluster_brand_id__num_like_u2i_30d,f_num__socdem_cluster_brand_id__num_like_i2i_30d,f_num__socdem_cluster_brand_id__num_like_catalog_30d,f_num__socdem_cluster_brand_id__num_like_search_30d,f_num__socdem_cluster_brand_id__num_like_other_30d,f_num__socdem_cluster_brand_id__conv_view_to_click_30d,f_num__socdem_cluster_brand_id__conv_view_to_clickout_30d,f_num__socdem_cluster_brand_id__conv_view_to_like_30d,f_num__socdem_cluster_brand_id__conv_click_to_clickout_30d,f_num__socdem_cluster_brand_id__conv_click_to_like_30d,f_num__socdem_cluster_brand_id__conv_clickout_to_like_30d
u8,u64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
12,137356,387.0,13.0,0.0,0.0,230.0,14.0,121.0,13.0,9.0,6.0,0.0,6.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.033592,0.0,0.0,0.0,0.0,null
17,19688,2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
4,39543,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
9,52917,2.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,null
10,190801,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
7,102766,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
4,176966,8.0,0.0,0.0,0.0,1.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null
19,173015,7.0,1.0,0.0,0.0,6.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.142857,0.0,0.0,0.0,0.0,null
20,211031,203.0,1.0,1.0,0.0,159.0,2.0,6.0,16.0,20.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.004926,0.004926,0.0,1.0,0.0,0.0


## II. Сбор датасета. (3 балла)

Будем учить модель ранжировать показанные пользователю рекомендации в рамках дня (можно было бы выбрать и другой промежуток). Разделим подготовку обучающих данных на два этапа: сбор "скелета" (базиса) с последующим созданием датасета путем джойна фичей на базис.

### II.I Сбор датасета -> Сбор базиса (1 из 3 баллов)

Базис представляется как (`session_id`, `user_id`, `item_id`, `label`), где в качестве `session_id` используется конкатенация `user_id` и `day`, а `label` зависит от `action_type`. Напомню, что view < click < clickout < like. Сессии, целиком состоящие из view, не стоит учитывать (действительно, сложно оценить качество сортировки одинаковых элементов). Если в рамках сессии было несколько взаимодействий с айтемом, то в качестве `label` нужно взять максимальное значение.

После того, как получим предсказания модели, можно будет сгруппировать базис по `session_id` и получить структуру ([`item_id`], [`label`], [`score`]) - тогда, отсортировав по айтемы по `label` либо `score` получим список айтемов с реальной либо модельной сортировкой, а по этому уже удобно считать метрики.

Реализуйте сбор базиса. Его так же удобно сохранять "за каждый день". Добавьте логику фильтрации сессий по 99 перцентилю длины. 

In [20]:
def build_basis(
    day_from: int,
    day_to: int,
    filter_99: bool = True,
    output_dir: Path = Path("output/basis/"),
) -> None:
    raw_dir = Path("datamart/raw/events/")
    output_dir.mkdir(parents=True, exist_ok=True)

    label_map = {
        "view": 0,
        "click": 1,
        "clickout": 2,
        "like": 3,
    }

    for day in tqdm(range(day_from, day_to + 1)):
        day_str = str(day).zfill(5)

        day_parts = []
        for action_type in ACTION_TYPES:
            path = raw_dir / action_type / f"{day_str}.pq"
            if path.exists():
                day_parts.append(
                    pl.scan_parquet(path).select(["user_id", "item_id", "action_type"])
                )

        if not day_parts:
            continue

        day_events = pl.concat(day_parts).with_columns([
            pl.format("{}_{}", pl.col("user_id"), pl.lit(day_str)).alias("session_id"),
            pl.col("action_type").replace_strict(label_map).cast(pl.Int8).alias("label"),
        ])

        basis_day = (
            day_events
            .group_by(["session_id", "user_id", "item_id"])
            .agg(pl.col("label").max().alias("label"))
        )

        basis_day = basis_day.join(
            basis_day.group_by("session_id").agg(pl.col("label").max().alias("session_max_label")),
            on="session_id",
            how="left",
        ).filter(pl.col("session_max_label") > 0).drop("session_max_label")

        if filter_99:
            session_lens = basis_day.group_by("session_id").agg(pl.len().alias("session_len"))
            p99 = session_lens.select(pl.col("session_len").quantile(0.99)).collect().item()
            basis_day = basis_day.join(session_lens, on="session_id", how="left").filter(
                pl.col("session_len") <= p99
            ).drop("session_len")

        basis_day.collect().write_parquet(output_dir / f"{day_str}.pq")

In [21]:
build_basis(day_from=min(DAYS), day_to=max(DAYS), filter_99=True)

100%|██████████| 51/51 [00:02<00:00, 24.34it/s]


In [22]:
basis = pl.read_parquet("output/basis/")
print(basis.shape)
basis.sample(10)

(1139294, 4)


session_id,user_id,item_id,label
str,u64,str,i8
"""13280199_01278""",13280199,"""nfmcg_10979844""",0
"""18302691_01298""",18302691,"""nfmcg_21623805""",1
"""78789974_01265""",78789974,"""nfmcg_18999775""",0
"""12904985_01292""",12904985,"""nfmcg_360767""",0
"""82374701_01270""",82374701,"""nfmcg_26836367""",0
"""27613975_01298""",27613975,"""nfmcg_1531769""",0
"""73407979_01297""",73407979,"""nfmcg_13140736""",0
"""3336934_01298""",3336934,"""nfmcg_6293367""",0
"""16049122_01300""",16049122,"""nfmcg_4260701""",0


In [23]:
for col in basis.columns:
    print(f"{col}: {basis[col].n_unique()}")

session_id: 38120
user_id: 15876
item_id: 19968
label: 3


In [24]:
basis["label"].value_counts().sort("label")

label,count
i8,u32
0,1037217
1,86900
2,15177


In [25]:
basis.group_by("session_id").agg(pl.len())["len"].describe()

statistic,value
str,f64
"""count""",38120.0
"""null_count""",0.0
"""mean""",29.887041
"""std""",31.095241
"""min""",1.0
"""25%""",8.0
"""50%""",20.0
"""75%""",40.0
"""max""",205.0


### II.II Сбор датасета -> Сбор датасета с фичами (2 из 3 баллов)

Чтобы создать датасет, достаточно приджойнить к базису фичи. Обратите внимание, что фичи должны быть собраны за предыдущий день, чтобы избежать ликов. То есть, если мы работаем с базисом на 1300 день, то фичи для него необходимо брать из 1299 дня. Помимо числовых, можно также добавить категориальные фичи.

Реализуйте необходимую логику. Добавьте возможность читать список фичей, которые необходимо приджойнить, из файла. 

In [26]:
def build_dataset(
    day_from: int,
    day_to: int,
    basis_dir: Path = Path("output/basis/"),
    output_dir: Path = Path("output/dataset/"),
    features_to_use_filepath: Path | None = None,
) -> None:

    output_dir.mkdir(parents=True, exist_ok=True)

    users = USERS.lazy()
    items = ITEMS.lazy()

    features_to_use = None
    if features_to_use_filepath is not None and Path(features_to_use_filepath).exists():
        features_to_use = [
            line.strip() for line in Path(features_to_use_filepath).read_text().splitlines()
            if line.strip()
        ]

    for day in tqdm(range(day_from, day_to + 1)):
        day_str = str(day).zfill(5)
        basis_path = basis_dir / f"{day_str}.pq"
        if not basis_path.exists():
            continue

        df = pl.scan_parquet(basis_path)
        df = join_features(
            df=df,
            day=day,
            users=users,
            items=items,
            features_to_use=features_to_use,
        )
        df.collect().write_parquet(output_dir / f"{day_str}.pq")

In [27]:
def join_features(
    df: pl.LazyFrame,
    day: int,
    users: pl.LazyFrame,
    items: pl.LazyFrame,
    features_dir: Path = Path("datamart/features/events/"),
    features_to_use: list[str] | None = None,
) -> pl.LazyFrame:

    user_cols = set(users.collect_schema().names())
    item_cols = set(items.collect_schema().names())

    join_user_cols = ["user_id"]
    for col in ["region", "socdem_cluster"]:
        if col in user_cols:
            join_user_cols.append(col)

    join_item_cols = ["item_id"]
    for col in ["brand_id", "category", "subcategory"]:
        if col in item_cols:
            join_item_cols.append(col)

    if len(join_user_cols) > 1:
        df = df.join(users.select(join_user_cols), on="user_id", how="left")
        for col in join_user_cols:
            if col != "user_id":
                df = df.with_columns(
                    pl.col(col).cast(pl.Utf8).alias(_feature_name(col, ["user_id"], type="cat"))
                )

    if len(join_item_cols) > 1:
        df = df.join(items.select(join_item_cols), on="item_id", how="left")
        for col in join_item_cols:
            if col != "item_id":
                df = df.with_columns(
                    pl.col(col).cast(pl.Utf8).alias(_feature_name(col, ["item_id"], type="cat"))
                )

    feature_day = day - 1
    if feature_day < min(DAYS):
        # Для самого раннего дня в датасете исторических событийных фич еще нет.
        return df
    feature_day_str = str(feature_day).zfill(5)

    for group_dir in sorted(features_dir.iterdir()):
        if not group_dir.is_dir():
            continue
        feat_path = group_dir / f"{feature_day_str}.pq"
        if not feat_path.exists():
            continue

        keys = group_dir.name.split("-")

        feat_lf = pl.scan_parquet(feat_path)
        feat_cols = feat_lf.collect_schema().names()

        keep_cols = keys.copy()
        if features_to_use is None:
            keep_cols.extend([c for c in feat_cols if c.startswith("f_")])
        else:
            keep_cols.extend([c for c in feat_cols if c in set(features_to_use)])

        keep_cols = [c for c in keep_cols if c in feat_cols]
        feat_lf = feat_lf.select(keep_cols)
        df = df.join(feat_lf, on=keys, how="left")

    return df


def build_dataset(
    day_from: int,
    day_to: int,
    basis_dir: Path = Path("output/basis/"),
    output_dir: Path = Path("output/dataset/"),
    features_to_use_filepath: Path | None = None,
    users: pl.LazyFrame | None = None,
    items: pl.LazyFrame | None = None,
) -> None:

    output_dir.mkdir(parents=True, exist_ok=True)

    if users is None:
        users = USERS.lazy()
    if items is None:
        items = ITEMS.lazy()

    features_to_use = None
    if features_to_use_filepath is not None and Path(features_to_use_filepath).exists():
        features_to_use = [
            line.strip() for line in Path(features_to_use_filepath).read_text().splitlines()
            if line.strip()
        ]

    for day in tqdm(range(day_from, day_to + 1)):
        day_str = str(day).zfill(5)
        basis_path = basis_dir / f"{day_str}.pq"
        if not basis_path.exists():
            continue

        day_df = pl.scan_parquet(basis_path)
        day_df = join_features(
            df=day_df,
            day=day,
            users=users,
            items=items,
            features_to_use=features_to_use,
        )
        day_df.collect().write_parquet(output_dir / f"{day_str}.pq")

In [28]:
build_dataset(day_from=min(DAYS), day_to=max(DAYS), output_dir = Path("output/dataset/"))

100%|██████████| 51/51 [00:06<00:00,  8.48it/s]


In [29]:
pl.read_parquet("output/dataset/01250.pq").sample(10)

session_id,user_id,item_id,label,region,socdem_cluster,f_cat__user_id__region,f_cat__user_id__socdem_cluster,brand_id,category,subcategory,f_cat__item_id__brand_id,f_cat__item_id__category,f_cat__item_id__subcategory
str,u64,str,i8,u8,u8,str,str,u64,str,str,str,str,str
"""53913309_01250""",53913309,"""nfmcg_24473567""",0,82,20,"""82""","""20""",137311,"""Fashion Accessories, Tech Add-…","""Jewelry and Costume Jewelry""","""137311""","""Fashion Accessories, Tech Add-…","""Jewelry and Costume Jewelry"""
"""5494773_01250""",5494773,"""nfmcg_3453441""",0,61,20,"""61""","""20""",95689,null,null,"""95689""",null,null
"""80970095_01250""",80970095,"""nfmcg_17182939""",0,2,10,"""2""","""10""",39543,null,null,"""39543""",null,null
"""33554032_01250""",33554032,"""nfmcg_243982""",1,63,0,"""63""","""0""",137356,"""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …","""137356""","""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …"
"""81472503_01250""",81472503,"""nfmcg_27799335""",0,66,4,"""66""","""4""",117885,null,null,"""117885""",null,null
"""48539316_01250""",48539316,"""nfmcg_11546245""",0,21,12,"""21""","""12""",137356,"""Miscellaneous Goods (Uncategor…",null,"""137356""","""Miscellaneous Goods (Uncategor…",null
"""4708641_01250""",4708641,"""nfmcg_22274857""",0,90,17,"""90""","""17""",23188,"""Miscellaneous Goods (Uncategor…",null,"""23188""","""Miscellaneous Goods (Uncategor…",null
"""41822425_01250""",41822425,"""nfmcg_22580280""",0,84,12,"""84""","""12""",23188,"""Household Electrical Appliance…","""Heaters, Air Conditioners, and…","""23188""","""Household Electrical Appliance…","""Heaters, Air Conditioners, and…"
"""71836997_01250""",71836997,"""nfmcg_5529683""",0,11,20,"""11""","""20""",238130,"""Home/Office Furniture and Inte…","""Cabinets and Storage Systems""","""238130""","""Home/Office Furniture and Inte…","""Cabinets and Storage Systems"""


## III Обучение ранжирования (4 балла)

### III.I. Подготовка выборок для обучения/валидации/теста и реализация метрик (1 из 4 баллов)

Полезно будет также написать функцию, считывающую с диска и возвращающую train, val, train_val, и test части датасета. Диапазон будем задавать через дни.

In [30]:
def read_dataset(
    day_from: int,
    n_train_days: int,
    n_val_days: int,
    n_test_days: int,
    dataset_dir: Path = Path("output/dataset/"),
) -> tuple[pl.LazyFrame, pl.LazyFrame, pl.LazyFrame, pl.LazyFrame]:
    train_days = list(range(day_from, day_from + n_train_days))
    val_start = day_from + n_train_days
    val_days = list(range(val_start, val_start + n_val_days))
    test_start = val_start + n_val_days
    test_days = list(range(test_start, test_start + n_test_days))

    def _read_days(days: list[int]) -> pl.LazyFrame:
        paths = [dataset_dir / f"{str(day).zfill(5)}.pq" for day in days if (dataset_dir / f"{str(day).zfill(5)}.pq").exists()]
        if not paths:
            return pl.DataFrame(schema={"session_id": pl.Utf8, "user_id": pl.UInt64, "item_id": pl.Utf8, "label": pl.Int8}).lazy()
        return pl.concat([pl.scan_parquet(path) for path in paths])

    train_df = _read_days(train_days)
    val_df = _read_days(val_days)
    test_df = _read_days(test_days)
    train_val_df = pl.concat([train_df, val_df])

    return train_df, val_df, train_val_df, test_df

In [31]:
train_df, val_df, train_val_df, test_df = read_dataset(
    day_from=1265,
    n_train_days=14,
    n_val_days=1,
    n_test_days=1,
    dataset_dir=Path("output/dataset/")
)
train_df = train_df.collect()
val_df = val_df.collect()
train_val_df = train_val_df.collect()
test_df = test_df.collect()

In [32]:
train_val_df.shape

(129033, 314)

In [33]:
test_df.shape

(8876, 314)

Реализуйте логику расчета метрик по структуре ([`item_id`], [`label`], [`score`]) - можете формировать эту структуру также внутри функции расчета метрик, а можете вне. В качестве метрики обязательно использовать NDCG@k. В выборе остальных метрик вы свободны. Полезно может быть считать метрики в разрезе по label - например, сколько айтемов с label=2 попали в топ-10 рекомендаций.

Посчитайте метрики в A/A-сеттинге.

In [34]:
class AtKMetric(ABC):
    def __init__(self, k: int):
        self.k = k

    @property
    @abstractmethod
    def name(self) -> str:
        raise NotImplementedError

    @property
    def full_name(self) -> str:
        return f"{self.name}@{self.k}"

    @abstractmethod
    def __call__(self, *, labels_col: str = "labels", targets_col: str = "targets") -> pl.Expr:
        raise NotImplementedError


class NdcgAtK(AtKMetric):
    @property
    def name(self) -> str:
        return "ndcg"

    def __call__(self, *, labels_col: str = "labels", targets_col: str = "targets") -> pl.Expr:
        k = self.k

        def _ndcg_at_k(row: dict) -> float:
            preds = row.get(labels_col) or []
            targets = row.get(targets_col) or []

            rel_by_item = {x["item_id"]: x["label"] for x in targets}
            pred_rels = [float(rel_by_item.get(x["item_id"], 0.0)) for x in preds[:k]]
            ideal_rels = sorted([float(x["label"]) for x in targets], reverse=True)[:k]

            dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(pred_rels))
            idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_rels))
            return float(dcg / idcg) if idcg > 0 else 0.0

        return pl.struct([labels_col, targets_col]).map_elements(_ndcg_at_k, return_dtype=pl.Float64)


def evaluate_ranker(
    df: pl.DataFrame,
    ks: list[int] = [1,  5, 10, 20, 50],
    preds_col: str = "preds",
    targets_col: str = "targets",
) -> pl.DataFrame:
    if isinstance(df, pl.LazyFrame):
        df_lf = df
    else:
        df_lf = df.lazy()

    metrics = {}
    for k in ks:
        metric = NdcgAtK(k=k)
        value = (
            df_lf
            .select(metric(labels_col=preds_col, targets_col=targets_col).mean().alias(metric.full_name))
            .collect()
            .item()
        )
        metrics[metric.full_name] = value

    return metrics

In [35]:
metrics_best = evaluate_ranker(
    test_df.with_columns(score=pl.col("label"))
    .group_by("session_id").agg(
        [
            pl.struct("item_id").sort_by("score", descending=True).alias("preds"),
            pl.struct("item_id", "label").sort_by("label", descending=True).alias("targets"),
        ]
    )
)
metrics_worst = evaluate_ranker(
    test_df.with_columns(score=pl.col("label"))
    .group_by("session_id").agg(
        [
            pl.struct("item_id").sort_by("score", descending=False).alias("preds"),
            pl.struct("item_id", "label").sort_by("label", descending=True).alias("targets"),
        ]
    )
)
RESULTS = pd.concat([
    pd.DataFrame(metrics_best, index=["best"]),
    pd.DataFrame(metrics_worst, index=["worst"]), 
])
RESULTS.style.format(precision=5).background_gradient(cmap="Blues")

,ndcg@1,ndcg@5,ndcg@10,ndcg@20,ndcg@50
best,1.00000,1.00000,1.00000,1.00000,1.00000
worst,0.12615,0.21220,0.30185,0.36900,0.41927


### III.II. Реализация TopPopular бейзлайна (1 из 4 баллов)

Реализуйте любой бейзлайн (бейзлайны) на своё усмотрение. Посчитайте метрики. Не забудьте про консистентность: учимся на train - оцениваем на val; учимся на train+val - оцениваем на test.

Важно: используйте `sample(fraction=1.0, shuffle=True)` при группировке по сессии для расчета метрик, чтобы в случае одинаковых скоров автоматом не проставлялся скор из корркетно отсортированной последовтаельности айтемов! 

In [36]:
item_popularity_train = (
    train_df
    .group_by("item_id")
    .agg(pl.len().alias("item_popularity"))
)

baseline_val = evaluate_ranker(
    val_df
    .join(item_popularity_train, on="item_id", how="left")
    .with_columns(pl.col("item_popularity").fill_null(0.0))
    .sample(fraction=1.0, shuffle=True)
    .group_by("session_id")
    .agg(
        [
            pl.struct("item_id").sort_by("item_popularity", descending=True).alias("preds"),
            pl.struct("item_id", "label").sort_by("label", descending=True).alias("targets"),
        ]
    )
)

item_popularity_train_val = (
    train_val_df
    .group_by("item_id")
    .agg(pl.len().alias("item_popularity"))
)

baseline_test = evaluate_ranker(
    test_df
    .join(item_popularity_train_val, on="item_id", how="left")
    .with_columns(pl.col("item_popularity").fill_null(0.0))
    .sample(fraction=1.0, shuffle=True)
    .group_by("session_id")
    .agg(
        [
            pl.struct("item_id").sort_by("item_popularity", descending=True).alias("preds"),
            pl.struct("item_id", "label").sort_by("label", descending=True).alias("targets"),
        ]
    )
)

RESULTS = pd.concat([
    RESULTS,
    pd.DataFrame(baseline_val, index=["baseline_val"]),
    pd.DataFrame(baseline_test, index=["baseline_test"]),
])
RESULTS.style.format(precision=5).background_gradient(cmap="Blues")

,ndcg@1,ndcg@5,ndcg@10,ndcg@20,ndcg@50
best,1.00000,1.00000,1.00000,1.00000,1.00000
worst,0.12615,0.21220,0.30185,0.36900,0.41927
baseline_val,0.29438,0.44866,0.52170,0.56373,0.58955
baseline_test,0.32689,0.49378,0.56804,0.60398,0.62225


In [37]:
test_df['user_id'].n_unique()

543

### III.III. Реализация обучения градиентного бустинга (2 из 4 баллов)

Обучите ранкер на train части датасета. В качестве модели можете использовать любую из [Catboost](https://catboost.ai), [LGBM](https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.Booster.html), [XGBoost](https://xgboost.readthedocs.io/en/stable/). Обучать можно как Ranker, так и Classifier, так и Regressor. Поэкспериментируйте. Посчитайте метрики на val части и подберите гиперпараметры (можете взять разные временные срезы, чтобы не заоверфиттиться под один).

Обучите итоговую модель на train + val, замерьте качество на test и сравните с бейзлайном.

Sanity check. Обратите внимание, что если вы считаете бейзлайн по фиче из датасета, то фича, по которой вы считаете бейзлайн, обязательно должна присутствовать как фича для ранкера. Если ранкер при использовании этой фичи показывает результаты хуже, чем бейзлайн, то что-то с вашим ранкером не так.

In [38]:
features = [col for col in train_df.columns if col.startswith("f_")]
categorical_features = [col for col in features if col.startswith("f_cat__")]

In [39]:
def train_evaluate_model(params, train_df, val_df, features, categorical_features):
    train = train_df.clone()
    val = val_df.clone()

    numeric_dtypes = {
        pl.Int8, pl.Int16, pl.Int32, pl.Int64,
        pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
        pl.Float32, pl.Float64,
    }

    detected_cat = [f for f in features if train.schema.get(f) not in numeric_dtypes]
    categorical_features = sorted(set(categorical_features) | set(detected_cat))
    num_features = [f for f in features if f not in categorical_features]

    for col in num_features:
        train = train.with_columns(pl.col(col).fill_null(0.0).cast(pl.Float32))
        val = val.with_columns(pl.col(col).fill_null(0.0).cast(pl.Float32))

    for col in categorical_features:
        train = train.with_columns(pl.col(col).cast(pl.Utf8).fill_null("UNK"))
        val = val.with_columns(pl.col(col).cast(pl.Utf8).fill_null("UNK"))

        cat_values = train.select(col).unique().to_series().to_list()
        cat_map = {v: i for i, v in enumerate(cat_values)}

        train = train.with_columns(pl.col(col).replace(cat_map, default=-1).cast(pl.Int32))
        val = val.with_columns(pl.col(col).replace(cat_map, default=-1).cast(pl.Int32))

    X_train = train.select(features).to_numpy()
    X_val = val.select(features).to_numpy()
    y_train = train["label"].to_numpy()

    cat_idx = [features.index(col) for col in categorical_features]

    model = lgbm.LGBMRegressor(**params)
    model.fit(X_train, y_train, categorical_feature=cat_idx)

    val_pred = model.predict(X_val)
    val_scored = val.with_columns(score=pl.Series(val_pred))

    metrics = evaluate_ranker(
        val_scored
        .sample(fraction=1.0, shuffle=True)
        .group_by("session_id")
        .agg(
            [
                pl.struct("item_id").sort_by("score", descending=True).alias("preds"),
                pl.struct("item_id", "label").sort_by("label", descending=True).alias("targets"),
            ]
        )
    )

    return model, metrics

def objective(trial):
    params = {
        "objective": "regression",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "num_leaves": trial.suggest_int("num_leaves", 31, 255),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1,
    }

    _, metrics = train_evaluate_model(params, train_df, val_df, features, categorical_features)
    return metrics["ndcg@5"]


study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=15)
print(f"\nBest ndcg@5: {study.best_value:.5f}")

[I 2026-04-24 12:28:17,466] A new study created in memory with name: no-name-42e76853-5105-43ac-9df1-f82f4add3a7a


/var/folders/4g/7w3zz2lj1d5bwq_jd175wjtw0000gn/T/ipykernel_62592/3304934967.py:26: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  train = train.with_columns(pl.col(col).replace(cat_map, default=-1).cast(pl.Int32))
/var/folders/4g/7w3zz2lj1d5bwq_jd175wjtw0000gn/T/ipykernel_62592/3304934967.py:27: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  val = val.with_columns(pl.col(col).replace(cat_map, default=-1).cast(pl.Int32))
/Users/maksimlunin/Desktop/jupyter/Рекомендательные системы/venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-04-24 12:29:00,589] Trial 0 finished with value: 0.4423093413260967 a


Best ndcg@5: 0.47938


In [40]:
best_params = study.best_params | {
    "objective": "regression",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}

model, _ = train_evaluate_model(best_params, train_val_df, test_df, features, categorical_features)

train_full = train_val_df.clone()
test_ready = test_df.clone()

numeric_dtypes = {
    pl.Int8, pl.Int16, pl.Int32, pl.Int64,
    pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
    pl.Float32, pl.Float64,
}
detected_cat = [f for f in features if train_full.schema.get(f) not in numeric_dtypes]
all_categorical = sorted(set(categorical_features) | set(detected_cat))
num_features = [f for f in features if f not in all_categorical]

for col in num_features:
    train_full = train_full.with_columns(pl.col(col).fill_null(0.0).cast(pl.Float32))
    test_ready = test_ready.with_columns(pl.col(col).fill_null(0.0).cast(pl.Float32))

for col in all_categorical:
    train_full = train_full.with_columns(pl.col(col).cast(pl.Utf8).fill_null("UNK"))
    test_ready = test_ready.with_columns(pl.col(col).cast(pl.Utf8).fill_null("UNK"))

    cat_values = train_full.select(col).unique().to_series().to_list()
    cat_map = {v: i for i, v in enumerate(cat_values)}

    train_full = train_full.with_columns(pl.col(col).replace(cat_map, default=-1).cast(pl.Int32))
    test_ready = test_ready.with_columns(pl.col(col).replace(cat_map, default=-1).cast(pl.Int32))

X_test = test_ready.select(features).to_numpy()
test_pred = model.predict(X_test)

ranker = evaluate_ranker(
    test_ready
    .with_columns(score=pl.Series(test_pred))
    .sample(fraction=1.0, shuffle=True)
    .group_by("session_id")
    .agg(
        [
            pl.struct("item_id").sort_by("score", descending=True).alias("preds"),
            pl.struct("item_id", "label").sort_by("label", descending=True).alias("targets"),
        ]
    )
)

RESULTS = pd.concat([
    RESULTS,
    pd.DataFrame(ranker, index=["ranker"]),
])
RESULTS.style.format(precision=5).background_gradient(cmap="Blues")

/var/folders/4g/7w3zz2lj1d5bwq_jd175wjtw0000gn/T/ipykernel_62592/3304934967.py:26: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  train = train.with_columns(pl.col(col).replace(cat_map, default=-1).cast(pl.Int32))
/var/folders/4g/7w3zz2lj1d5bwq_jd175wjtw0000gn/T/ipykernel_62592/3304934967.py:27: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  val = val.with_columns(pl.col(col).replace(cat_map, default=-1).cast(pl.Int32))
/Users/maksimlunin/Desktop/jupyter/Рекомендательные системы/venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/var/folders/4g/7w3zz2lj1d5bwq_jd175wjtw0000gn/T/ipykernel_62592/162386245.py

,ndcg@1,ndcg@5,ndcg@10,ndcg@20,ndcg@50
best,1.00000,1.00000,1.00000,1.00000,1.00000
worst,0.12615,0.21220,0.30185,0.36900,0.41927
baseline_val,0.29438,0.44866,0.52170,0.56373,0.58955
baseline_test,0.32689,0.49378,0.56804,0.60398,0.62225
ranker,0.36280,0.51778,0.59638,0.62843,0.64271


Получилось обогнать baseline бустингом на тесте по всем ndcg@k метрикам. Ключевая причина состоит в том, что baseline учитывает только глобальную популярность айтема, тогда как бустинг использует персонализированные и контекстные признаки из датамарта (история взаимодействий по пользователю, айтему и их сочетанию, конверсии, категориальные признаки и т.д.). Это позволяет лучше расставлять айтемы внутри конкретной сессии пользователя. Кроме этого, для повышения качества проводился тюнинг гиперпараметров на валидации.